# Build 05 — Randomisation Strategy: Epsilon-Greedy & Thompson Sampling

> *Active exploration policies to break the SFP loop*

**EN:** Pure model policies create the loop. By forcibly randomising a fraction of investigations (ε-greedy) or sampling from Beta posteriors (Thompson), we generate counterfactual labels that both unbias evaluation and improve future training data.

**KR:** 순수 모델 정책이 루프를 만듦. 일부 조사를 강제로 무작위화(ε-greedy)하거나 Beta posterior에서 샘플링(Thompson)하면, 평가도 unbias하고 미래 학습 데이터도 개선하는 counterfactual 라벨 생성.

## 1. Imports & FEATURE_COLS

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, recall_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


FEATURE_COLS = ['high_amount', 'night_claim', 'high_postcode', 'prior_claims']

## 2. Policy Base + Pure Model + Epsilon-Greedy

In [ ]:
class InvestigationPolicy:
    """Base class for investigation policies."""
    name: str = "Base"

    def decide(
        self,
        scores: np.ndarray,
        investigation_budget: float,
        rng: np.random.Generator
    ) -> np.ndarray:
        """Return binary array: 1 = investigate this claim, 0 = skip."""
        raise NotImplementedError


class PureModelPolicy(InvestigationPolicy):
    """Investigate top-k% claims by model score. No exploration."""
    name = "Pure Model (ε=0)"

    def decide(self, scores, investigation_budget, rng):
        n = len(scores)
        n_investigate = int(n * investigation_budget)
        threshold = np.sort(scores)[::-1][n_investigate - 1]
        return (scores >= threshold).astype(int)


class EpsilonGreedyPolicy(InvestigationPolicy):
    """
    Epsilon-greedy: with probability ε, investigate randomly;
    with probability (1-ε), follow model ranking.
    """
    def __init__(self, epsilon: float):
        self.epsilon = epsilon
        self.name = f"ε-Greedy (ε={epsilon:.2f})"

    def decide(self, scores, investigation_budget, rng):
        n = len(scores)
        n_investigate = int(n * investigation_budget)

        # Number of random investigations
        n_random = int(n_investigate * self.epsilon)
        n_model = n_investigate - n_random

        # Model-driven: top n_model by score
        top_model_idx = np.argsort(scores)[::-1][:n_model]

        # Random: uniform sample from remaining claims
        remaining_idx = np.setdiff1d(np.arange(n), top_model_idx)
        random_idx = rng.choice(remaining_idx, size=min(n_random, len(remaining_idx)), replace=False)

        investigated = np.zeros(n, dtype=int)
        investigated[top_model_idx] = 1
        investigated[random_idx] = 1
        return investigated

## 3. Thompson Sampling Policy

In [ ]:
class ThompsonSamplingPolicy(InvestigationPolicy):
    """
    Thompson Sampling: treat each claim as a Bernoulli bandit.
    Sample from Beta posterior for each claim; investigate top-k samples.
    (Simplified: uses model score as prior alpha parameter)
    """
    name = "Thompson Sampling"

    def decide(self, scores, investigation_budget, rng):
        n = len(scores)
        n_investigate = int(n * investigation_budget)

        # Beta parameters: alpha = score * 10 + 1, beta = (1-score) * 10 + 1
        alpha = scores * 10 + 1
        beta = (1 - scores) * 10 + 1

        # Sample from Beta distribution for each claim
        samples = rng.beta(alpha, beta)
        top_idx = np.argsort(samples)[::-1][:n_investigate]

        investigated = np.zeros(n, dtype=int)
        investigated[top_idx] = 1
        return investigated

## 4. Online Simulation Engine

In [ ]:
def run_policy_simulation(
    df_full: pd.DataFrame,
    policy: InvestigationPolicy,
    n_periods: int = 10,
    investigation_budget: float = 0.25,
    initial_seed_size: float = 0.05,
    investigator_fnr: float = 0.02,
    seed: int = 42,
) -> list[dict]:
    """
    Simulate online deployment of an investigation policy over n_periods.

    Each period:
        1. Score all claims with current model
        2. Apply policy → get investigation decisions
        3. Observe fraud only in investigated claims (+ random subset if ε > 0)
        4. Retrain model on accumulated biased (or partially unbiased) data
        5. Record metrics against ground truth

    Returns:
        List of per-period metric dicts
    """
    rng = np.random.default_rng(seed)
    n = len(df_full)
    period_size = n // n_periods

    # Initial unbiased seed model
    seed_size = int(n * initial_seed_size)
    seed_idx = rng.choice(n, size=seed_size, replace=False)
    seed_df = df_full.iloc[seed_idx].copy()
    seed_df['observed_fraud'] = seed_df['true_fraud']

    current_model = LogisticRegression(max_iter=1000, class_weight='balanced')
    X_seed = seed_df[FEATURE_COLS].values
    y_seed = seed_df['observed_fraud'].values
    current_model.fit(X_seed, y_seed)

    all_training_data = [seed_df]
    metrics_per_period = []

    for period in range(n_periods):
        start = period * period_size
        end = min(start + period_size, n)
        period_df = df_full.iloc[start:end].copy()

        if len(period_df) == 0:
            break

        # Score this period's claims
        X_period = period_df[FEATURE_COLS].values
        scores = current_model.predict_proba(X_period)[:, 1]

        # Apply policy
        investigated = policy.decide(scores, investigation_budget, rng)

        # Observe fraud in investigated claims
        observed_fraud = np.full(len(period_df), np.nan)
        invest_mask = investigated == 1
        true_fraud_period = period_df['true_fraud'].values

        observed_fraud[invest_mask] = np.where(
            true_fraud_period[invest_mask] == 1,
            (rng.random(invest_mask.sum()) > investigator_fnr).astype(int),
            0
        )

        period_df['model_score'] = scores
        period_df['investigated'] = investigated
        period_df['observed_fraud'] = observed_fraud

        # Compute metrics against TRUE ground truth
        true_fraud_all = df_full['true_fraud'].values
        model_scores_all = current_model.predict_proba(df_full[FEATURE_COLS].values)[:, 1]

        auc_true = roc_auc_score(true_fraud_all, model_scores_all)
        true_recall = recall_score(
            true_fraud_all,
            (model_scores_all >= 0.5).astype(int),
            zero_division=0
        )

        # Observed fraud rate in investigated claims (biased view)
        invest_mask_bool = invest_mask
        observed_fraud_rate = period_df.loc[invest_mask_bool, 'observed_fraud'].mean()

        # Blind spot: fraction of true fraud in uninvestigated claims (this period)
        blind_spot_fraud = (
            period_df.loc[~invest_mask_bool, 'true_fraud'] == 1
        ).sum()
        total_fraud_period = (period_df['true_fraud'] == 1).sum()
        blind_spot_fraction = blind_spot_fraud / max(total_fraud_period, 1)

        # Loop amplification
        top_q_mask = model_scores_all >= np.percentile(model_scores_all, 75)
        fraud_rate_top = true_fraud_all[top_q_mask].mean()
        true_fraud_rate = true_fraud_all.mean()
        loop_amp = fraud_rate_top / true_fraud_rate if true_fraud_rate > 0 else 1.0

        metrics_per_period.append({
            'period': period + 1,
            'policy': policy.name,
            'n_claims': len(period_df),
            'n_investigated': int(invest_mask.sum()),
            'investigation_rate': invest_mask.mean(),
            'observed_fraud_rate': round(observed_fraud_rate, 4),
            'auc_vs_true': round(auc_true, 4),
            'true_recall': round(true_recall, 4),
            'loop_amplification': round(loop_amp, 3),
            'blind_spot_fraction': round(blind_spot_fraction, 4),
        })

        # Retrain model on ALL accumulated data
        new_train = period_df.dropna(subset=['observed_fraud']).copy()
        new_train['observed_fraud'] = new_train['observed_fraud'].astype(int)
        all_training_data.append(new_train)
        combined = pd.concat(all_training_data, ignore_index=True)

        X_train = combined[FEATURE_COLS].values
        y_train = combined['observed_fraud'].values
        if len(np.unique(y_train)) > 1:
            current_model = LogisticRegression(max_iter=1000, class_weight='balanced')
            current_model.fit(X_train, y_train)

    return metrics_per_period

## 5. Cost-Benefit Analysis

In [ ]:
def cost_benefit_analysis(
    metrics_dict: dict[str, list[dict]],
    investigation_cost_per_claim: float = 150.0,  # £ per investigation
    fraud_value_recovered: float = 3000.0,         # £ per fraud detected
) -> pd.DataFrame:
    """
    Compute the financial cost-benefit of each exploration policy.

    Cost:   n_investigations × cost_per_investigation
    Benefit: n_fraud_detected × value_recovered

    For exploration policies, random investigations have lower hit rate
    but generate training data that improves future model recall.
    """
    rows = []
    for policy_name, metrics in metrics_dict.items():
        final_period = metrics[-1]
        avg_recall = np.mean([m['true_recall'] for m in metrics])
        final_recall = final_period['true_recall']
        avg_investigate_rate = np.mean([m['investigation_rate'] for m in metrics])
        total_n = sum(m['n_claims'] for m in metrics)
        total_investigated = sum(m['n_investigated'] for m in metrics)

        # Approximate: assume 8% true fraud rate across 50k claims
        assumed_fraud_rate = 0.08
        total_fraud_estimated = int(total_n * assumed_fraud_rate)

        total_cost = total_investigated * investigation_cost_per_claim
        total_fraud_detected = int(total_fraud_estimated * avg_recall)
        total_benefit = total_fraud_detected * fraud_value_recovered
        net_benefit = total_benefit - total_cost
        roi = (net_benefit / total_cost) * 100 if total_cost > 0 else 0

        rows.append({
            'Policy': policy_name,
            'Final Recall': f"{final_recall:.3f}",
            'Avg Recall (all periods)': f"{avg_recall:.3f}",
            'Total Investigations': f"{total_investigated:,}",
            'Total Cost (£)': f"£{total_cost:,.0f}",
            'Est. Fraud Detected': f"{total_fraud_detected:,}",
            'Total Benefit (£)': f"£{total_benefit:,.0f}",
            'Net Benefit (£)': f"£{net_benefit:,.0f}",
            'ROI (%)': f"{roi:.1f}%",
        })

    return pd.DataFrame(rows)

## 6. Policy Comparison Plot

In [ ]:
def plot_policy_comparison(
    metrics_dict: dict[str, list[dict]],
    metric: str = 'true_recall',
    title: str = None
) -> None:
    """Plot a metric over time for multiple policies."""
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['tomato', 'steelblue', 'seagreen', 'darkorchid', 'darkorange']

    for i, (policy_name, metrics) in enumerate(metrics_dict.items()):
        periods = [m['period'] for m in metrics]
        values = [m[metric] for m in metrics]
        ax.plot(periods, values, 'o-', label=policy_name,
                color=colors[i % len(colors)], linewidth=2, markersize=6)

    metric_labels = {
        'true_recall': 'True Recall (vs ground truth)',
        'loop_amplification': 'Loop Amplification Factor',
        'blind_spot_fraction': 'Blind Spot Fraction (this period)',
        'auc_vs_true': 'AUC vs True Fraud Labels',
    }
    ylabel = metric_labels.get(metric, metric)

    ax.set_xlabel('Period')
    ax.set_ylabel(ylabel)
    ax.set_title(title or f'{ylabel} Over Periods')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    fname = f'policy_comparison_{metric}.png'
    plt.savefig(fname, dpi=150)
    plt.show()
    print(f"Saved: {fname}")

---

# 👁️ Section A — Hands-on: 5 Policies, Head to Head

> *We generate 20k claims and run 8 periods under each policy. The output is one giant comparison.*

**EN:** Watch how recall evolves period-by-period. Pure model plateaus or falls. Epsilon-greedy variants recover; the more exploration, the more recall (but the more cost). Thompson sits between.

**KR:** period마다 recall 변화 관찰. Pure model은 정체/하락. ε-greedy 계열은 회복; ε이 클수록 recall ↑(비용도 ↑). Thompson은 중간.

## A.1 Generate data

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve().parent / '01_sfp_simulation'))
from simulate_sfp_loop import generate_claims

print("Generating 20k claims with ground-truth fraud labels...")
df = generate_claims(n_claims=20_000, seed=42)

print(f"\n📊 Data ready:")
print(f"  Total claims      : {len(df):,}")
print(f"  True fraud rate   : {df['true_fraud'].mean():.3f}")
print(f"  Total true frauds : {df['true_fraud'].sum():,}")

## A.2 Run all 5 policies

**EN:** Pure model + ε ∈ {0.05, 0.10, 0.20} + Thompson Sampling. 8 sequential periods each. Prints final-period metrics as each policy completes.

**KR:** Pure model + ε ∈ {0.05, 0.10, 0.20} + Thompson Sampling. 각 8개 period. 정책 완료마다 최종 메트릭 출력.

In [ ]:
policies = [
    PureModelPolicy(),
    EpsilonGreedyPolicy(epsilon=0.05),
    EpsilonGreedyPolicy(epsilon=0.10),
    EpsilonGreedyPolicy(epsilon=0.20),
    ThompsonSamplingPolicy(),
]

all_metrics = {}

for policy in policies:
    print(f"\n{'─'*60}")
    print(f"Simulating: {policy.name}")
    print(f"{'─'*60}")
    metrics = run_policy_simulation(
        df_full=df, policy=policy,
        n_periods=8, investigation_budget=0.25,
    )
    all_metrics[policy.name] = metrics
    final = metrics[-1]
    print(f"  ✅ Final TRUE recall      : {final['true_recall']:.4f}")
    print(f"     Final loop amplification: {final['loop_amplification']:.2f}×")
    print(f"     Final blind spot         : {final['blind_spot_fraction']:.1%}")

## A.3 Per-period summary table

In [ ]:
for policy_name, metrics in all_metrics.items():
    print(f"\n📊 {policy_name}")
    summary = pd.DataFrame(metrics)
    print(summary[['period', 'investigation_rate', 'true_recall',
                   'loop_amplification', 'blind_spot_fraction']].to_string(index=False))

## A.4 Side-by-side final recall comparison

In [ ]:
print(f"\n{'Policy':<26} {'Final recall':<14} {'Final loop amp':<16} {'Final blind spot'}")
print("─" * 80)
for policy_name, metrics in all_metrics.items():
    final = metrics[-1]
    print(f"  {policy_name:<24} {final['true_recall']:<14.4f} "
          f"{final['loop_amplification']:<16.2f} {final['blind_spot_fraction']:.1%}")

## A.5 Cost-benefit (financial framing)

In [ ]:
print()
cb_df = cost_benefit_analysis(
    all_metrics,
    investigation_cost_per_claim=150.0,
    fraud_value_recovered=3000.0,
)
print(cb_df.to_string(index=False))

## A.6 Visual: recall over time

In [ ]:
plot_policy_comparison(all_metrics, metric='true_recall',
                       title='True Recall Recovery Under 5 Policies')
plot_policy_comparison(all_metrics, metric='loop_amplification',
                       title='Loop Amplification Under 5 Policies')

## 🎯 What you should observe

**EN:**
- **Pure model**: recall flat or declining across periods → loop dominates
- **ε = 0.05**: small but real recall recovery
- **ε = 0.20**: largest recall gain but cost rises
- **Thompson Sampling**: balances exploration and exploitation automatically — competitive with ε=0.10
- **ROI**: depends on £150/investigation × £3000/fraud assumption — adjust to Allianz numbers

**KR:**
- **Pure model**: period 간 recall 정체/하락 → 루프 지배
- **ε = 0.05**: 작지만 실제 recall 회복
- **ε = 0.20**: 가장 큰 recall 이득, 비용도 상승
- **Thompson Sampling**: 탐색과 활용을 자동 균형 — ε=0.10 수준
- **ROI**: £150/조사 × £3000/fraud 가정에 의존 — Allianz 숫자로 조정 필요